In [1]:
import geopandas as gpd 
import pandas as pd 

latlon_json = "/data01/FDS/muduchuru/Atmos/NEXGDDP_HYRAS_BC_CSV/latlon_to_rowcol.json"
gpkg_soil = "/beegfs/muduchuru/pkgs_fnl/climdata/usecase/sdba/Site_Soil_BZE_WGS84.gpkg"

gdf = gpd.read_file(gpkg_soil)

In [3]:
gdf.columns

Index(['PointID', 'County', 'Sampling_m', 'Sampling_y', 'Soil_clima',
       'Land_use', 'BZE_peat', 'Main_soil', 'Specific_s', 'Groundwate',
       'Groundwa_1', 'Thickness', 'Thicknes_1', 'Slope', 'Exposition',
       'Curvature', 'Type_of_re', 'Position_i', 'CS_0_30', 'CS_30_100',
       'Longitude', 'Latitude', 'NUTS_ID', 'NUTS_NAME', 'STATE_NAME',
       'geometry'],
      dtype='object')

In [5]:
df = pd.read_json(latlon_json)

In [ ]:
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

# -----------------------------
# Extract grid information
# -----------------------------
grid_coords = np.array(df[0].tolist())   # [[lat, lon], ...]
grid_indices = np.array(df[1].tolist())  # [[row, col], ...]

# Build KDTree for fast nearest-neighbor search
tree = cKDTree(grid_coords)

# -----------------------------
# Query nearest grid point
# -----------------------------
point_coords = gdf[['Latitude', 'Longitude']].to_numpy()

distance, nearest_idx = tree.query(point_coords, k=1)

nearest_latlon = grid_coords[nearest_idx]
nearest_rowcol = grid_indices[nearest_idx]

# -----------------------------
# Create output dataframe
# -----------------------------
output = gdf[
    [
        'PointID',
        'NUTS_ID',
        'NUTS_NAME',
        'STATE_NAME',
        'Latitude',
        'Longitude'
    ]
].copy()

output['nearest_grid_id'] = [
    f"C{col}R{row}"
    for row, col in nearest_rowcol
]

output['nearest_latitude'] = nearest_latlon[:, 0]
output['nearest_longitude'] = nearest_latlon[:, 1]

# (optional) distance in degrees
output['distance_deg'] = distance

# -----------------------------
# Save CSV
# -----------------------------
# output.to_csv("point_to_nearest_grid.csv", index=False)

print(output.head())

   PointID NUTS_ID      NUTS_NAME          STATE_NAME   Latitude  Longitude  \
0        2   DEF07  Nordfriesland  Schleswig-Holstein  54.859923   8.411608   
1        3   DEF07  Nordfriesland  Schleswig-Holstein  54.864382   8.697143   
2        4   DEF07  Nordfriesland  Schleswig-Holstein  54.866979   8.765298   
3        5   DEF07  Nordfriesland  Schleswig-Holstein  54.863920   8.959032   
4        6   DEF07  Nordfriesland  Schleswig-Holstein  54.870685   9.078456   

  nearest_grid_id  nearest_latitude  nearest_longitude  distance_deg  
0         C325R23             54.86               8.71      0.298392  
1         C325R23             54.86               8.71      0.013583  
2         C331R23             54.86               8.77      0.008415  
3         C347R23             54.86               8.93      0.029295  
4         C362R27             54.82               9.08      0.050709  
